# Tarea Hogar 04 — cambiar los datos, no los hiperparámetros (z498)

Experimento **HT4980**. z497 mostró que los hiperparámetros tocaron techo: el top 10 de la etapa 2 cae entre 568M y 584M con error estándar de ~7M, y el ganador sacó 348–350 en Kaggle, contra 375.9 de KA5940.

Dos hallazgos explican la diferencia:

1. **Label.** KA5940 (z494, la cátedra) entrena con `clase01 = BAJA+1 o BAJA+2`. z495, z496 y z497 entrenaban solo con BAJA+2. Localmente, con la config del centro y 6 semillas pareadas, BAJA+1+2 da +13M (ganó en 5 de 6).
2. **Drift 202107 → 202109.** Un modelo que adivina el mes lo logra siempre. `*_fultimo_cierre` vale 3 en julio y 1 en septiembre; sin esas columnas igual adivina con AUC 0.97 (`*_Fvencimiento`, límites, montos). Las variables que más usa el modelo son montos en pesos: la mediana de los saldos bajó ~20% y la de payroll subió ~16%.

Escalera de variantes, **cada una suma un cambio a la anterior**:

| variante | cambio |
|---|---|
| V0 | igual a z497 (BAJA+2, variables crudas) |
| V1 | + label BAJA+1 y BAJA+2 |
| V2 | + sin columnas de fecha (`*_fultimo_cierre`, `*_Fvencimiento`) |
| V3 | + montos en rank dentro de cada mes (cero fijo) |
| V4 | + variables nuevas dentro del mes (uso de tarjeta, saldo/payroll, ...) |

Una sola config (el centro de z497). Para cada variante: holdout local con semillas pareadas, y un modelo final con semillerío que genera el CSV.

**Ojo:** V2 y V3 atacan el cambio de mes, y eso el holdout de 202107 no lo puede ver. Se deciden en Kaggle, con el mismo cupo para todas. Diferencias de menos de ~10 puntos en el Public son ruido.

In [ ]:
if (!require("data.table")) install.packages("data.table")
if (!require("lightgbm")) install.packages("lightgbm")
require("data.table")
require("lightgbm")
require("parallel")

setDTthreads(percent = 100)
options(scipen = 999)

## 1. PARAM

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211L
PARAM$estudiante <- "Maceo, Marcos"
PARAM$experimento <- "HT4980"
PARAM$mc_cores <- max(1L, detectCores() - 1L)

PARAM$test_frac <- 0.30
PARAM$ventana <- 200L
# +++ semillas del holdout: las mismas para todas las variantes, asi la comparacion es pareada
PARAM$semillas_holdout <- c(271211L, 200177L, 410551L, 552581L, 892237L, 123457L, 654323L, 777011L)
PARAM$semillerio <- 5L          # +++ modelos promediados en el final de CADA variante

PARAM$variantes <- c("V0", "V1", "V2", "V3", "V4")
PARAM$cupos_csv <- seq(9500L, 12500L, by = 500L)
PARAM$cupo_kaggle <- 11000L     # +++ el MISMO cupo para todas: solo cambia la variante
PARAM$variantes_kaggle <- c("V1", "V2", "V3", "V4")   # +++ V0 ya sabemos que anda ~350

# +++ config: el centro de z497 (1000 arboles, 8 hojas). No se tunea aca.
PARAM$lgb <- list(
  objective = "binary", metric = "auc", boosting = "gbdt", verbosity = -1,
  learning_rate = 0.027, num_leaves = 8L, max_depth = -1L, min_data_in_leaf = 76L,
  feature_fraction = 0.8, bagging_fraction = 1.0, bagging_freq = 0L,
  lambda_l1 = 0, lambda_l2 = 0, min_gain_to_split = 0, min_sum_hessian_in_leaf = 0.001,
  feature_pre_filter = FALSE, force_row_wise = TRUE
)
PARAM$num_iterations <- 1000L
PARAM$max_bin <- 127L
PARAM$mc_cores

## 2. Dataset

In [ ]:
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) {
  base_exp <- candidatos_exp[3]
  dir.create(base_exp, recursive = TRUE, showWarnings = FALSE)
}
dir.create(file.path(base_exp, PARAM$experimento), showWarnings = FALSE)
setwd(file.path(base_exp, PARAM$experimento))
getwd()

candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))

dataset <- fread(archivo_dataset)
dataset[, intersect(c("clase01", "azar", "training"), names(dataset)) := NULL]
dataset_mes <- dataset[foto_mes == 202107]
dataset[, .N, .(foto_mes, clase_ternaria)]

## 3. Variantes

`preparar(dt, variante)` devuelve las columnas de entrada para esa variante. Todo se calcula **dentro de cada mes** (`by = foto_mes`), así que da lo mismo aplicarlo a 202107 solo o a los dos meses juntos.

- **Rank con cero fijo:** los positivos van a (0, 1], los negativos a [-1, 0) y el 0 queda en 0. Un "saldo en el 90% de arriba" significa lo mismo en julio que en septiembre, aunque haya inflación. El 0 importa porque "no tiene el producto" es información.
- **Variables nuevas:** los cocientes (uso de la tarjeta, saldo / payroll) no dependen de la inflación. Las sumas Visa + Master empiezan con `m` y, en V4, también pasan por el rank.

In [ ]:
FECHAS <- c("Visa_fultimo_cierre", "Master_fultimo_cierre", "Visa_Fvencimiento", "Master_Fvencimiento")
NO_FEATURE <- c("clase_ternaria", "numero_de_cliente", "foto_mes")

es_monto <- function(nm) grepl("^m|^Visa_m|^Master_m", nm)

rank_cero_fijo <- function(x) {
  o <- rep(NA_real_, length(x))
  p <- which(x > 0)
  n <- which(x < 0)
  o[p] <- data.table::frank(x[p], ties.method = "average") / length(p)
  o[n] <- -data.table::frank(-x[n], ties.method = "average") / length(n)
  o[which(x == 0)] <- 0
  o
}

div0 <- function(a, b) ifelse(is.na(b) | b == 0, NA_real_, a / b)

agregar_fe <- function(dt) {
  s <- function(...) rowSums(cbind(...), na.rm = TRUE)
  dt[, mtc_consumo := s(Visa_mconsumototal, Master_mconsumototal)]
  dt[, mtc_limite := s(Visa_mlimitecompra, Master_mlimitecompra)]
  dt[, mtc_saldo := s(Visa_msaldototal, Master_msaldototal)]
  dt[, mtc_pagado := s(Visa_mpagado, Master_mpagado)]
  dt[, ctc_consumos := s(Visa_cconsumos, Master_cconsumos)]
  dt[, r_tc_uso := div0(mtc_consumo, mtc_limite)]
  dt[, r_tc_saldo_limite := div0(mtc_saldo, mtc_limite)]
  dt[, r_tc_pago_saldo := div0(mtc_pagado, mtc_saldo)]
  dt[, r_saldo_payroll := div0(mcuentas_saldo, mpayroll)]
  dt[, r_consumo_payroll := div0(mtc_consumo, mpayroll)]
  dt[, r_rentab_saldo := div0(mrentabilidad, abs(mcuentas_saldo))]
  dt[, r_comis_trx := div0(mcomisiones, ctrx_quarter)]
  dt[, r_ctrx_quarter_norm := ctrx_quarter / pmin(pmax(cliente_antiguedad, 1), 3)]
  dt[, r_caja_ahorro_share := div0(mcaja_ahorro, mcuentas_saldo)]
  invisible(dt)
}

preparar <- function(dt, variante) {
  k <- match(variante, c("V0", "V1", "V2", "V3", "V4"))
  x <- data.table::copy(dt)
  if (k >= 5) agregar_fe(x)
  if (k >= 3) x[, intersect(FECHAS, names(x)) := NULL]
  if (k >= 4) {
    montos <- setdiff(names(x)[es_monto(names(x))], NO_FEATURE)
    montos <- montos[vapply(x[, ..montos], is.numeric, logical(1))]
    x[, (montos) := lapply(.SD, rank_cero_fijo), by = foto_mes, .SDcols = montos]
  }
  campos <- setdiff(names(x), NO_FEATURE)
  campos <- campos[vapply(x[, ..campos], is.numeric, logical(1))]
  list(X = data.matrix(x[, ..campos]), campos = campos)
}

label_de <- function(clase, variante) {
  if (variante == "V0") as.integer(clase == "BAJA+2") else as.integer(clase %in% c("BAJA+1", "BAJA+2"))
}

for (v in PARAM$variantes) cat(v, ":", length(preparar(dataset_mes[1:1000], v)$campos), "columnas\n")

## 4. Holdout local, pareado por semilla

Mismo split 70/30 estratificado para todas las variantes en cada semilla. La métrica es el máximo de la curva de ganancia suavizada (±200 clientes), igual que z497. Lo que importa es la **diferencia contra la variante anterior en la misma semilla**, no el valor suelto.

In [ ]:
eval_holdout <- function(job) {
  tryCatch({
    data.table::setDTthreads(1)
    t0 <- Sys.time()
    set.seed(job$semilla)
    dw <- data.table::copy(dataset_mes)
    dw[, `:=`(azar_ = runif(.N), rowid_ = seq_len(.N))]
    data.table::setorderv(dw, c("clase_ternaria", "azar_"))
    dw[, fold_ := seq_len(.N) / .N, by = clase_ternaria]
    ids_tr <- dw[fold_ <= 1 - PARAM$test_frac, rowid_]
    ids_te <- setdiff(seq_len(nrow(dataset_mes)), ids_tr)

    prep <- preparar(dataset_mes, job$variante)
    y <- label_de(dataset_mes$clase_ternaria, job$variante)
    ds <- lightgbm::lgb.Dataset(prep$X[ids_tr, ], label = y[ids_tr],
                                params = list(max_bin = PARAM$max_bin, feature_pre_filter = FALSE))
    m <- lightgbm::lgb.train(modifyList(PARAM$lgb, list(num_threads = 1L, seed = job$semilla)),
                             ds, nrounds = PARAM$num_iterations, verbose = -1)
    prob <- predict(m, prep$X[ids_te, ])
    clase <- dataset_mes$clase_ternaria[ids_te]
    cs <- cumsum(ifelse(clase[order(-prob)] == "BAJA+2", 975000, -25000))
    suave <- data.table::frollmean(cs, 2L * PARAM$ventana + 1L, align = "center")
    i <- which.max(suave)
    imp <- lightgbm::lgb.importance(m)
    data.table::data.table(
      variante = job$variante, semilla = job$semilla,
      ganancia = suave[i] / PARAM$test_frac, envios = as.integer(round(i / PARAM$test_frac)),
      top = paste(head(imp$Feature, 8), collapse = ", "),
      tiempo_seg = round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1), error = ""
    )
  }, error = function(e) data.table::data.table(variante = job$variante, semilla = job$semilla,
                                                ganancia = NA_real_, error = conditionMessage(e)))
}

archivo_hold <- "holdout_th04_z498.tsv"
jobs <- lapply(seq_len(length(PARAM$variantes) * length(PARAM$semillas_holdout)), function(i) {
  g <- expand.grid(variante = PARAM$variantes, semilla = PARAM$semillas_holdout, stringsAsFactors = FALSE)
  as.list(g[i, ])
})
tb_hold <- if (file.exists(archivo_hold)) fread(archivo_hold, sep = "\t")[is.finite(ganancia)] else data.table()
hechos <- if (nrow(tb_hold)) paste(tb_hold$variante, tb_hold$semilla) else character()
pending <- Filter(function(j) !(paste(j$variante, j$semilla) %in% hechos), jobs)
cat("pendientes:", length(pending), "de", length(jobs), "\n")

if (length(pending)) {
  cl <- makeCluster(PARAM$mc_cores, type = "PSOCK")
  clusterEvalQ(cl, {
    Sys.setenv(OMP_NUM_THREADS = "1")
    suppressPackageStartupMessages({ library(data.table); library(lightgbm) })
    data.table::setDTthreads(1)
    NULL
  })
  clusterExport(cl, c("eval_holdout", "preparar", "agregar_fe", "rank_cero_fijo", "div0", "es_monto", "label_de",
                      "FECHAS", "NO_FEATURE", "PARAM", "dataset_mes"), envir = .GlobalEnv)
  t0 <- Sys.time()
  bs <- PARAM$mc_cores
  for (b in seq_len(ceiling(length(pending) / bs))) {
    idx <- ((b - 1L) * bs + 1L):min(b * bs, length(pending))
    nuevo <- rbindlist(parLapply(cl, pending[idx], eval_holdout), fill = TRUE)
    tb_hold <- rbindlist(list(tb_hold, nuevo), fill = TRUE)
    fwrite(tb_hold, archivo_hold, sep = "\t")
    cat(sprintf("tanda %d | ok %d | fallos %d | %.1f min\n", b, nuevo[is.finite(ganancia), .N],
                nuevo[!is.finite(ganancia), .N], as.numeric(difftime(Sys.time(), t0, units = "mins"))))
    if (nuevo[!is.finite(ganancia), .N]) print(nuevo[!is.finite(ganancia), .(variante, error)])
    flush.console()
  }
  stopCluster(cl)
}

In [ ]:
ok <- tb_hold[is.finite(ganancia)]
res <- ok[, .(ganancia_M = round(mean(ganancia) / 1e6, 1), se_M = round(sd(ganancia) / sqrt(.N) / 1e6, 1),
              envios = as.integer(median(envios)), n = .N), by = variante][order(variante)]

# +++ pareado: cada variante contra la anterior, en la misma semilla
w <- dcast(ok, semilla ~ variante, value.var = "ganancia")
vs <- PARAM$variantes
res[, delta_vs_anterior_M := NA_real_]
res[, se_delta_M := NA_real_]
res[, gana_semillas := NA_character_]
for (i in seq_along(vs)[-1]) {
  d <- (w[[vs[i]]] - w[[vs[i - 1]]]) / 1e6
  d <- d[is.finite(d)]
  res[variante == vs[i], `:=`(delta_vs_anterior_M = round(mean(d), 1),
                              se_delta_M = round(sd(d) / sqrt(length(d)), 1),
                              gana_semillas = sprintf("%d/%d", sum(d > 0), length(d)))]
}
fwrite(res, "resumen_th04_z498.tsv", sep = "\t")
print(res)
cat("\nV3 da IGUAL que V2 en el holdout: el rank dentro del mes no cambia el orden, los arboles son los mismos.\nV2 y V3 atacan el cambio de mes; empate en el holdout no es mala señal. Los decide Kaggle.\n\n")
for (v in vs) cat(v, "| top features:", ok[variante == v, top][1], "\n")

## 5. Modelo final por variante, CSV y Kaggle

Cada variante entrena sobre **todo** 202107, sin undersampling, con `min_data_in_leaf / 0.7`: el holdout veía el 70% del mes y el final ve el 100%. Semillerío: se promedian las probabilidades de `PARAM$semillerio` semillas.

Se graban los CSV de todos los cupos de `PARAM$cupos_csv`. A Kaggle se sube **un solo cupo por variante** (`PARAM$cupo_kaggle`), así la única diferencia entre subidas es la variante.

Si una sube más de ~10 puntos contra la anterior, el cambio sirve. Después se barre el cupo solo para la mejor.

In [ ]:
CORRER_KAGGLE <- FALSE   # +++ TRUE para subir PARAM$variantes_kaggle al cupo PARAM$cupo_kaggle

dfut <- dataset[foto_mes == 202109]
semillas_final <- PARAM$semillas_holdout[seq_len(PARAM$semillerio)]
min_data_final <- as.integer(round(PARAM$lgb$min_data_in_leaf / (1 - PARAM$test_frac)))
cat("min_data_in_leaf final:", min_data_final, "| semillerio:", semillas_final, "\n")

for (v in PARAM$variantes) {
  archivo_prob <- sprintf("KA498_%s_prob.csv", v)
  if (file.exists(archivo_prob)) {
    cat(v, ": ya estaba, la reuso\n")
    tb_prob <- fread(archivo_prob)
  } else {
    t0 <- Sys.time()
    todo <- rbindlist(list(dataset_mes, dfut), fill = TRUE)
    prep <- preparar(todo, v)
    es_tr <- todo$foto_mes == 202107
    y <- label_de(todo$clase_ternaria[es_tr], v)
    ds <- lgb.Dataset(prep$X[es_tr, ], label = y, params = list(max_bin = PARAM$max_bin, feature_pre_filter = FALSE),
                      free_raw_data = FALSE)
    prob <- numeric(sum(!es_tr))
    for (s in semillas_final) {
      m <- lgb.train(modifyList(PARAM$lgb, list(num_threads = PARAM$mc_cores + 1L, seed = s,
                                                min_data_in_leaf = min_data_final)),
                     ds, nrounds = PARAM$num_iterations, verbose = -1)
      prob <- prob + predict(m, prep$X[!es_tr, ]) / length(semillas_final)
    }
    tb_prob <- data.table(numero_de_cliente = todo$numero_de_cliente[!es_tr], prob = prob)
    fwrite(tb_prob, archivo_prob)
    cat(sprintf("%s: %d columnas, %.1f min\n", v, length(prep$campos), as.numeric(difftime(Sys.time(), t0, units = "mins"))))
    rm(todo, prep, ds); gc(verbose = FALSE)
  }
  ord <- order(-tb_prob$prob)
  for (envios in PARAM$cupos_csv) {
    pred <- integer(nrow(tb_prob))
    pred[ord[seq_len(envios)]] <- 1L
    fwrite(data.table(numero_de_cliente = tb_prob$numero_de_cliente, Predicted = pred),
           sprintf("KA498_%s_%05d.csv", v, envios))
  }
  if (CORRER_KAGGLE && v %in% PARAM$variantes_kaggle) {
    archivo <- sprintf("KA498_%s_%05d.csv", v, PARAM$cupo_kaggle)
    linea <- sprintf("kaggle competitions submit -c labo-1-ba-inicial -f %s -m 'z498 %s envios=%d semillerio=%d'",
                     archivo, v, PARAM$cupo_kaggle, length(semillas_final))
    cat("  SUBMIT", v, ":", system(linea, intern = TRUE), "\n")
    Sys.sleep(10)
  }
  flush.console()
}

# +++ cuanto se parecen los rankings entre variantes (top cupo_kaggle en comun)
tops <- lapply(PARAM$variantes, function(v) {
  p <- fread(sprintf("KA498_%s_prob.csv", v))
  p[order(-prob)][seq_len(PARAM$cupo_kaggle), numero_de_cliente]
})
names(tops) <- PARAM$variantes
solap <- outer(seq_along(tops), seq_along(tops), Vectorize(function(i, j) length(intersect(tops[[i]], tops[[j]])) / PARAM$cupo_kaggle))
dimnames(solap) <- list(PARAM$variantes, PARAM$variantes)
round(solap, 3)

In [ ]:
if (CORRER_KAGGLE) {
  Sys.sleep(30)
  cat(system("kaggle competitions submissions -c labo-1-ba-inicial", intern = TRUE)[1:12], sep = "\n")
}